In [212]:
import numpy as np
import pandas as pd
import os
import re
import nltk
from nltk.stem.porter import PorterStemmer


## A Tale of Two Cities

In [213]:
OHCO = ["book_num",'chap_num', 'para_num', 'sent_num', 'token_num']

In [214]:
base_path='/Users/josephhudson/Desktop/DS 5001 Project/Dickens Corpus/'

In [215]:
book_names=["A Tale of Two Cities.txt","Barnaby Rudge.txt","Bleak House.txt",
           "David Copperfield.txt","Dombey and Son.txt","Great Expectations.txt",
           "Hard Times.txt","Little Dorrit.txt","Martin Chuzzlewit.txt","Nicholas Nickleby.txt",
           "Oliver Twist.txt","Our Mutual Friend.txt","Pickwick Papers.txt",
           "The Mystery of Edwin Drood.txt","The Old Curiosity Shop.txt"]

In [216]:
books = {}
char_counts = []

for title in book_names:
    file_path = os.path.join(base_path,title)
    with open(file_path, "r", encoding="utf-8") as f:
        books[title] = pd.DataFrame(f.readlines(), columns=['line_str'])


for title in book_names:
    file_path = os.path.join(base_path,title)
    with open(file_path, "r", encoding="utf-8") as f:
        text=f.read()
        char_counts.append({
        'book_title': title.replace('.txt',''),
        'n_chars': len(text)})

Corpus_Chars = pd.DataFrame(char_counts)

In [217]:
#Avg length of each document in characters
Corpus_Chars["n_chars"].mean()

np.float64(1448075.2)

In [218]:
def clean_text_multi_book(LINES):
    OHCO = ['book_num', 'chap_num', 'para_num', 'sent_num', 'token_num']

    LINES = LINES.copy()
    LINES.index.name = 'line_num'
    LINES['line_str'] = (
        LINES['line_str']
        .str.replace(r'\n+', '', regex=True)
        .str.strip()
    )

    start_pat = r"\*\*\* START OF"
    end_pat   = r"\*\*\* END OF"

    start = LINES[LINES.line_str.str.contains(start_pat, case=False, na=False)].index[0] + 1
    end   = LINES[LINES.line_str.str.contains(end_pat, case=False, na=False)].index[0] - 1

    LINES = LINES.loc[start:end].copy()

    book_pat =r'^\s*BOOK\s+THE\s+(FIRST|SECOND|THIRD)\b'

    book_lines = LINES.line_str.str.match(book_pat, case=False, na=False)

    LINES.loc[book_lines, 'book_num'] = range(1, book_lines.sum() + 1)

    LINES['book_num'] = LINES['book_num'].ffill()

    chap_pat = r'^\s*Chapter\s+([IVXLCDM]+|\d+)\b'

    LINES['chap_flag'] = LINES.line_str.str.match(chap_pat, case=False, na=False)

    LINES['chap_num'] = (
        LINES.groupby('book_num')['chap_flag']
        .cumsum()
    )
    LINES.loc[(LINES['chap_num'] == 0), 'chap_num'] = np.nan

    LINES['chap_num'] = (
        LINES.groupby('book_num')['chap_num']
        .ffill()
    )

    LINES = LINES.dropna(subset=['book_num', 'chap_num'])

    LINES = LINES.loc[~book_lines]
    LINES = LINES.loc[~LINES['chap_flag']]

    LINES['book_num'] = LINES['book_num'].astype(int)
    LINES['chap_num'] = LINES['chap_num'].astype(int)

    CHAPS = (
        LINES.groupby(OHCO[:2])['line_str']
        .apply(lambda x: '\n'.join(x))
        .to_frame('chap_str')
    )
    PARAS = (
        CHAPS['chap_str']
        .str.split(r'\n\s*\n', expand=True)
        .stack()
        .to_frame('para_str')
        .sort_index()
    )

    PARAS.index.names = OHCO[:3]

    PARAS['para_str'] = PARAS['para_str'].str.strip()

    PARAS = PARAS[PARAS['para_str'] != '']
    SENTS = (
        PARAS['para_str']
        .str.split(r'[.!?;:]+', expand=True)
        .stack()
        .to_frame('sent_str')
        .sort_index()
    )

    SENTS.index.names = OHCO[:4]

    SENTS['sent_str'] = SENTS['sent_str'].str.strip()

    SENTS = SENTS[SENTS['sent_str'] != '']

    TOKENS = (
        SENTS['sent_str']
        .str.split(r"[\s',\-]+", expand=True)
        .stack()
        .to_frame('token_str')
        .sort_index()
    )

    TOKENS.index.names = OHCO

    TOKENS['term_str'] = (
        TOKENS['token_str']
        .str.replace(r'[\W_]+', '', regex=True)
        .str.lower()
    )

    TOKENS = TOKENS[TOKENS['term_str'] != '']

    VOCAB = (
        TOKENS['term_str']
        .value_counts()
        .to_frame('n')
        .reset_index()
        .rename(columns={'index': 'term_str'})
    )

    VOCAB.index.name = 'term_id'

    return TOKENS, VOCAB

In [219]:
tokens_dict = {}
vocab_dict = {}

In [220]:
title="A Tale of Two Cities.txt"

In [221]:
TOKENS, VOCAB = clean_text_multi_book(books["A Tale of Two Cities.txt"])

TOKENS = TOKENS.reset_index()

TOKENS['book_num'] = TOKENS['book_num'].replace({
    4: "A Tale of Two Cities 1",
    5: "A Tale of Two Cities 2",
    6: "A Tale of Two Cities 3"
})

TOKENS = TOKENS.set_index(OHCO)
    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.index = TOKENS.index.set_names('book_title', level=0)

clean_tale_two_cities_tokens= TOKENS
clean_tale_two_cities_vocab = VOCAB

In [222]:
clean_tale_two_cities_tokens

token_str term_str
book_title             chap_num para_num sent_num token_num                   
A Tale of Two Cities 1 1        0        0        0               The      the
                                                  1            Period   period
                                1        0        0                It       it
                                                  1               was      was
                                                  2               the      the
...                                                               ...      ...
A Tale of Two Cities 3 15       51       1        11             than     than
                                                  12                I        i
                                                  13             have     have
                                                  14             ever     ever
                                                  15            known    known

[137076 rows x 2 columns]

## Our Mututal friend

In [223]:
def clean_text_multi_book(LINES):

    OHCO = ['book_num', 'chap_num', 'para_num', 'sent_num', 'token_num']

    LINES = LINES.copy()
    LINES.index.name = 'line_num'
    LINES['line_str'] = (
        LINES['line_str']
        .str.replace(r'\n+', '', regex=True)
        .str.strip()
    )
    start_pat = r"\*\*\* START OF"
    end_pat   = r"\*\*\* END OF"

    start = LINES[LINES.line_str.str.contains(start_pat, case=False, na=False)].index[0] + 1
    end   = LINES[LINES.line_str.str.contains(end_pat, case=False, na=False)].index[0] - 1

    LINES = LINES.loc[start:end].copy()

    book_pat = r'^\s*Book\s+the\s+(First|Second|Third|Fourth)\b'

    book_lines = LINES.line_str.str.match(book_pat, case=False, na=False)

    LINES.loc[book_lines, 'book_num'] = range(1, book_lines.sum() + 1)

    LINES['book_num'] = LINES['book_num'].ffill()

    chap_pat = r'^\s*Chapter\s+([IVXLCDM]+|\d+)\b'

    LINES['chap_flag'] = LINES.line_str.str.match(chap_pat, case=False, na=False)

    LINES['chap_num'] = (
        LINES.groupby('book_num')['chap_flag']
        .cumsum()
    )

    LINES.loc[(LINES['chap_num'] == 0), 'chap_num'] = np.nan

    LINES['chap_num'] = (
        LINES.groupby('book_num')['chap_num']
        .ffill()
    )


    LINES = LINES.dropna(subset=['book_num', 'chap_num'])

    LINES = LINES.loc[~book_lines]
    LINES = LINES.loc[~LINES['chap_flag']]

    LINES['book_num'] = LINES['book_num'].astype(int)
    LINES['chap_num'] = LINES['chap_num'].astype(int)

    CHAPS = (
        LINES.groupby(OHCO[:2])['line_str']
        .apply(lambda x: '\n'.join(x))
        .to_frame('chap_str')
    )

    PARAS = (
        CHAPS['chap_str']
        .str.split(r'\n\s*\n', expand=True)
        .stack()
        .to_frame('para_str')
        .sort_index()
    )

    PARAS.index.names = OHCO[:3]

    PARAS['para_str'] = PARAS['para_str'].str.strip()

    PARAS = PARAS[PARAS['para_str'] != '']

    SENTS = (
        PARAS['para_str']
        .str.split(r'[.!?;:]+', expand=True)
        .stack()
        .to_frame('sent_str')
        .sort_index()
    )

    SENTS.index.names = OHCO[:4]

    SENTS['sent_str'] = SENTS['sent_str'].str.strip()

    SENTS = SENTS[SENTS['sent_str'] != '']

    TOKENS = (
        SENTS['sent_str']
        .str.split(r"[\s',\-]+", expand=True)
        .stack()
        .to_frame('token_str')
        .sort_index()
    )

    TOKENS.index.names = OHCO

    TOKENS['term_str'] = (
        TOKENS['token_str']
        .str.replace(r'[\W_]+', '', regex=True)
        .str.lower()
    )

    TOKENS = TOKENS[TOKENS['term_str'] != '']

    VOCAB = (
        TOKENS['term_str']
        .value_counts()
        .to_frame('n')
        .reset_index()
        .rename(columns={'index': 'term_str'})
    )

    VOCAB.index.name = 'term_id'

    return TOKENS, VOCAB

In [224]:
title="Our Mutual Friend.txt"

In [225]:
TOKENS, VOCAB = clean_text_multi_book(books[title])

TOKENS = TOKENS.reset_index()

TOKENS['book_num'] = TOKENS['book_num'].replace({
    5: "Our Mutual Friend 1",
    6: "Our Mutual Friend 2",
    7: "Our Mutual Friend 3",
    8: "Our Mutual Friend 4"
})

TOKENS = TOKENS.set_index(OHCO)
    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.index = TOKENS.index.set_names('book_title', level=0)

clean_mutual_friend_tokens= TOKENS
clean_mutual_friend_vocab = VOCAB

In [226]:
clean_mutual_friend_tokens.index.get_level_values('book_title').value_counts()

book_title
Our Mutual Friend 1    83928
Our Mutual Friend 2    82882
Our Mutual Friend 3    82574
Our Mutual Friend 4    77491
Name: count, dtype: int64

In [227]:
clean_mutual_friend_tokens

token_str   term_str
book_title          chap_num para_num sent_num token_num                      
Our Mutual Friend 1 1        0        0        0                 ON         on
                                               1                THE        the
                                               2               LOOK       look
                                               3                OUT        out
                             1        0        0                 In         in
...                                                             ...        ...
Our Mutual Friend 4 17       64       5        0               —THE        the
                                               1                END        end
                             65       0        0          September  september
                                               1                2nd        2nd
                                               2               1865       1865

[326875 rows x 2 columns]

## Bardany Rudge

In [228]:
OHCO= ['chap_num', 'para_num', 'sent_num', 'token_num']

In [229]:
def clean_text(LINES, chap_pat):

    OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

    LINES = LINES.copy()
    LINES.index.name = 'line_num'

    LINES['line_str'] = (
        LINES['line_str']
        .str.replace(r'\n+', ' ', regex=True)
        .str.strip()
    )


    clip_pats = [
        r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
        r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
    ]

    pat_a = LINES.line_str.str.match(clip_pats[0], case=False, na=False)
    pat_b = LINES.line_str.str.match(clip_pats[1], case=False, na=False)

    line_a = LINES.loc[pat_a].index[0] + 1
    line_b = LINES.loc[pat_b].index[0] - 1

    LINES = LINES.loc[line_a:line_b].copy()

    chap_lines = LINES.line_str.str.match(chap_pat, case=False, na=False)

    LINES.loc[chap_lines, 'chap_num'] = range(1, chap_lines.sum() + 1)

    LINES['chap_num'] = LINES['chap_num'].ffill()

    LINES = LINES.dropna(subset=['chap_num'])

    LINES = LINES.loc[~chap_lines]

    LINES['chap_num'] = LINES['chap_num'].astype(int)

    CHAPS = (
        LINES.groupby(OHCO[:1])['line_str']
        .apply(lambda x: '\n'.join(x))
        .to_frame('chap_str')
    )

    CHAPS['chap_str'] = CHAPS['chap_str'].str.strip()

    para_pat = r'\n\n+'

    PARAS = (
        CHAPS['chap_str']
        .str.split(para_pat, expand=True)
        .stack()
        .to_frame('para_str')
        .sort_index()
    )

    PARAS.index.names = OHCO[:2]

    PARAS['para_str'] = (
        PARAS['para_str']
        .astype(str)
        .str.replace(r'\n', ' ', regex=True)
        .str.strip()
    )

    PARAS = PARAS[~PARAS['para_str'].str.match(r'^\s*$')]

    sent_pat = r'[.?!;:]+'

    SENTS = (
        PARAS['para_str']
        .str.split(sent_pat, expand=True)
        .stack()
        .dropna()
        .to_frame('sent_str')
    )

    SENTS.index.names = OHCO[:3]

    SENTS['sent_str'] = SENTS['sent_str'].astype(str).str.strip()

    SENTS = SENTS[~SENTS['sent_str'].str.match(r'^\s*$')]

    token_pat = r"[\s',-]+"

    TOKENS = (
        SENTS['sent_str']
        .str.split(token_pat, expand=True)
        .stack()
        .dropna()
        .to_frame('token_str')
    )

    TOKENS.index.names = OHCO

    TOKENS['token_str'] = TOKENS['token_str'].astype(str)

    TOKENS['term_str'] = (
        TOKENS['token_str']
        .str.replace(r'[\W_]+', '', regex=True)
        .str.lower()
    )

    TOKENS = TOKENS[TOKENS['term_str'] != '']

    VOCAB = (
        TOKENS['term_str']
        .value_counts()
        .to_frame('n')
        .reset_index()
        .rename(columns={'index': 'term_str'})
    )

    VOCAB.index.name = 'term_id'

    return TOKENS, VOCAB

In [230]:
title="Barnaby Rudge.txt"

In [231]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r"^\s*(?:chapter|letter)\s+(?:\d+|the\s+last)$")

    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Barnaby Rudge"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])

clean_barnaby_tokens= TOKENS
clean_barnaby_vocab = VOCAB

In [232]:
clean_barnaby_tokens

token_str term_str
book_title    chap_num para_num sent_num token_num                   
Barnaby Rudge 1        0        0        0                In       in
                                         1               the      the
                                         2              year     year
                                         3              1775     1775
                                         4             there    there
...                                                      ...      ...
              82       20       1        20          talking  talking
                                         21               to       to
                                         22              the      the
                                         23          present  present
                                         24             time     time

[255395 rows x 2 columns]

## Bleak House

In [233]:
title="Bleak House.txt"

In [234]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r"^\s*(?:chapter|letter)\s+(?:\d+|[ivxlcdm]+|the\s+last)$")

    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Bleak House"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])

clean_bleak_tokens= TOKENS
clean_bleak_vocab = VOCAB

In [235]:
clean_bleak_tokens

token_str    term_str
book_title  chap_num para_num sent_num token_num                        
Bleak House 1        0        0        0                  In          in
                                       1            Chancery    chancery
                     1        0        0              London      london
                              1        0          Michaelmas  michaelmas
                                       1                term        term
...                                                      ...         ...
            67       28       2        48               much        much
                                       49             beauty      beauty
                                       50                 in          in
                                       51            me—even      meeven
                                       52         supposing—   supposing

[354643 rows x 2 columns]

## David Copperfield

In [236]:
title="David Copperfield.txt"

In [237]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r"^\s*(?:chapter|letter)\s+(?:\d+|[ivxlcdm]+|the\s+last)\b.*$")

    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "David Copperfield"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])

clean_david_tokens= TOKENS
clean_david_vocab = VOCAB

In [238]:
clean_david_tokens

token_str  term_str
book_title        chap_num para_num sent_num token_num                    
David Copperfield 1        0        0        0           Whether   whether
                                             1                 I         i
                                             2             shall     shall
                                             3              turn      turn
                                             4               out       out
...                                                          ...       ...
                  64       33       1        18             thee      thee
                                             19             near      near
                                             20               me        me
                                             21         pointing  pointing
                                             22           upward    upward

[357805 rows x 2 columns]

## Dombey and Son

In [239]:
title="Dombey and Son.txt"

In [240]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+(?:\d+|[ivxlcdm]+)\.\s*$')

    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Dombey and Son"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])

clean_dombey_tokens= TOKENS
clean_dombey_vocab = VOCAB

In [241]:
clean_dombey_tokens

token_str term_str
book_title     chap_num para_num sent_num token_num                   
Dombey and Son 1        0        0        0            Dombey   dombey
                                          1               and      and
                                          2               Son      son
                        1        0        0            Dombey   dombey
                                          1               sat      sat
...                                                       ...      ...
               62       67       2        51           friend   friend
                                          52              and      and
                                          53                I        i
                                          54           parted   parted
                                          55          company  company

[356587 rows x 2 columns]

## Great Expectations

In [242]:
title="Great Expectations.txt"

In [243]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*(?:chapter|letter)\s+(?:\d+|[ivxlcdm]+|the\s+last)\.?\s*$')

TOKENS = TOKENS.reset_index()

TOKENS = TOKENS[TOKENS['chap_num'] > 59]

TOKENS['chap_num'] = TOKENS['chap_num'].replace(
    dict(zip(range(60, 119), range(1, 60)))
)

TOKENS = TOKENS.set_index(OHCO)

    
tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB


TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Great Expectations"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])


clean_expectations_tokens= TOKENS
clean_expectations_vocab = VOCAB

clean_expectations_tokens

token_str term_str
book_title         chap_num para_num sent_num token_num                   
Great Expectations 1        0        0        0                My       my
                                              1          father’s  fathers
                                              2            family   family
                                              3              name     name
                                              4             being    being
...                                                           ...      ...
                   59       45       1        39               of       of
                                              40          another  another
                                              41          parting  parting
                                              42             from     from
                                              43              her      her

[186085 rows x 2 columns]

## Hard times

In [244]:
title="Hard Times.txt"

In [245]:
OHCO= ["book_num",'chap_num', 'para_num', 'sent_num', 'token_num']

In [246]:
TOKENS, VOCAB = clean_text_multi_book(books[title])


tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB


TOKENS.reset_index(inplace=True)

TOKENS['book_num'] = TOKENS['book_num'].replace({
    1: "Hard Times 1",
    2: "Hard Times 2",
    3: "Hard Times 3"
})

TOKENS = TOKENS.set_index(OHCO)

TOKENS.index = TOKENS.index.set_names('book_title', level=0)

clean_times_tokens= TOKENS
clean_times_vocab = VOCAB

clean_times_tokens

token_str  term_str
book_title   chap_num para_num sent_num token_num                    
Hard Times 1 1        0        0        0               THE       the
                                        1               ONE       one
                                        2             THING     thing
                                        3           NEEDFUL   needful
                      1        0        0              ‘NOW       now
...                                                     ...       ...
Hard Times 3 9        38       0        14              not       not
                                        15         included  included
                                        16               in        in
                                        17             this      this
                                        18            eText     etext

[103497 rows x 2 columns]

## Little Dorrit

In [247]:
OHCO= ["book_num",'chap_num', 'para_num', 'sent_num', 'token_num']

In [248]:
title="Little Dorrit.txt"

In [249]:
TOKENS, VOCAB = clean_text_multi_book(books[title])

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS['book_num'] = TOKENS['book_num'].replace({
    3: "Little Dorrit 1",
    4: "Little Dorrit 2",
})

TOKENS = TOKENS.set_index(OHCO)

TOKENS.index = TOKENS.index.set_names('book_title', level=0)


clean_dorrit_tokens= TOKENS
clean_dorrit_vocab = VOCAB

clean_dorrit_tokens

token_str    term_str
book_title      chap_num para_num sent_num token_num                        
Little Dorrit 1 1        1        0        0              Thirty      thirty
                                           1               years       years
                                           2                 ago         ago
                                           3          Marseilles  marseilles
                                           4                 lay         lay
...                                                          ...         ...
Little Dorrit 2 34       93       4        26                and         and
                                           27               made        made
                                           28              their       their
                                           29              usual       usual
                                           30             uproar      uproar

[340254 rows x 2 columns]

## Martin Chuzzlewit

In [250]:
OHCO= ['chap_num', 'para_num', 'sent_num', 'token_num']

In [251]:
title="Martin Chuzzlewit.txt"


In [252]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+.*$')

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Martin Chuzzlewit"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])



clean_chuzzle_tokens= TOKENS
clean_chuzzle_vocab = VOCAB

clean_chuzzle_tokens

token_str  \
book_title        chap_num para_num sent_num token_num                 
Martin Chuzzlewit 1        0        0        0          INTRODUCTORY   
                                             1            CONCERNING   
                                             2                   THE   
                                             3              PEDIGREE   
                                             4                    OF   
...                                                              ...   
                  55       105      2        24              uplifts   
                                             25                   ye   
                                             26                 both   
                                             27                   to   
                                             28               Heaven   

                                                            term_str  
book_title        chap_num para_num sent_num token_num                
Martin Chuzzlewit 1        0        0        0          introductory  
                                             1            concerning  
                                             2                   the  
                                             3              pedigree  
                                             4                    of  
...                                                              ...  
                  55       105      2        24              uplifts  
                                             25                   ye  
                                             26                 both  
                                             27                   to  
                                             28               heaven  

[338838 rows x 2 columns]

## Nichoas Nickleby

In [253]:
title="Nicholas Nickleby.txt"

In [254]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+.*$')

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Nicholas Nickleby"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])



clean_nickle_tokens= TOKENS
clean_nickle_vocab = VOCAB

clean_nickle_tokens

token_str    term_str
book_title        chap_num para_num sent_num token_num                        
Nicholas Nickleby 1        0        0        0          Introduces  introduces
                                             1                 all         all
                                             2                 the         the
                                             3                Rest        rest
                           1        0        0               There       there
...                                                            ...         ...
                  65       14       2        30                 of          of
                                             31              their       their
                                             32               poor        poor
                                             33               dead        dead
                                             34             cousin      cousin

[324259 rows x 2 columns]

## Oliver Twist

In [255]:
title ="Oliver Twist.txt"

In [256]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+.*$')

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Oliver Twist"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])



clean_twist_tokens= TOKENS
clean_twist_vocab = VOCAB

clean_twist_tokens

token_str term_str
book_title   chap_num para_num sent_num token_num                   
Oliver Twist 1        0        0        0            TREATS   treats
                                        1                OF       of
                                        2               THE      the
                                        3             PLACE    place
                                        4             WHERE    where
...                                                     ...      ...
             53       16       5        14              she      she
                                        15              was      was
                                        16             weak     weak
                                        17              and      and
                                        18           erring   erring

[158285 rows x 2 columns]

## Pick Wick Papers

In [257]:
OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

In [258]:
title="Pickwick Papers.txt"

In [259]:
def clean_text_pickwick(LINES):
    LINES = LINES.copy()
    LINES.index.name = "line_num"
    LINES["line_str"] = (
        LINES["line_str"]
        .astype(str)
        .str.replace(r"\r", " ", regex=True)
        .str.strip()
    )
    end_pat = r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"

    end_mask = LINES.line_str.str.contains(end_pat, case=False, na=False)

    if end_mask.any():
        end_line = LINES.index[end_mask][0] - 1
        LINES = LINES.loc[:end_line].copy()

    title_pat = r"THE POSTHUMOUS PAPERS OF THE PICKWICK CLUB"

    title_mask = LINES.line_str.str.contains(
        title_pat,
        case=False,
        na=False
    )

    title_hits = LINES.index[title_mask]

    if len(title_hits) >= 2:
        start_line = title_hits[1]
    elif len(title_hits) == 1:
        start_line = title_hits[0]

    LINES = LINES.loc[start_line:].copy()

    chap_pat = r"^\s*CHAPTER\s+[IVXLCDM]+\."

    chap_lines = LINES.line_str.str.match(
        chap_pat,
        case=False,
        na=False
    )

    LINES.loc[chap_lines, "chap_num"] = range(1, chap_lines.sum() + 1)

    LINES["chap_num"] = LINES["chap_num"].ffill()


    LINES = LINES.dropna(subset=["chap_num"]).copy()

    LINES = LINES.loc[~chap_lines].copy()

    LINES["chap_num"] = LINES["chap_num"].astype(int)
    CHAPS = (
        LINES.groupby("chap_num")["line_str"]
        .apply(lambda x: "\n".join(x))
        .to_frame("chap_str")
    )

    CHAPS["chap_str"] = CHAPS["chap_str"].str.strip()

    para_pat = r"\n\n+"

    PARAS = (
        CHAPS["chap_str"]
        .str.split(para_pat, expand=True)
        .stack()
        .to_frame("para_str")
        .sort_index()
    )

    PARAS.index.names = ["chap_num", "para_num"]

    PARAS["para_str"] = (
        PARAS["para_str"]
        .astype(str)
        .str.replace(r"\n", " ", regex=True)
        .str.strip()
    )

    PARAS = PARAS[~PARAS["para_str"].str.match(r"^\s*$")]

    sent_pat = r"[.?!;:]+"

    SENTS = (
        PARAS["para_str"]
        .str.split(sent_pat, expand=True)
        .stack()
        .dropna()
        .to_frame("sent_str")
    )

    SENTS.index.names = ["chap_num", "para_num", "sent_num"]

    SENTS["sent_str"] = SENTS["sent_str"].astype(str).str.strip()
    SENTS = SENTS[~SENTS["sent_str"].str.match(r"^\s*$")]


    token_pat = r"[\s',\-—]+"

    TOKENS = (
        SENTS["sent_str"]
        .str.split(token_pat, expand=True)
        .stack()
        .dropna()
        .to_frame("token_str")
    )

    TOKENS.index.names = [
        "chap_num",
        "para_num",
        "sent_num",
        "token_num"
    ]

    TOKENS["token_str"] = TOKENS["token_str"].astype(str)

    TOKENS["term_str"] = (
        TOKENS["token_str"]
        .str.replace(r"[\W_]+", "", regex=True)
        .str.lower()
    )

    TOKENS = TOKENS[TOKENS["term_str"] != ""]

    VOCAB = (
        TOKENS["term_str"]
        .value_counts()
        .to_frame("n")
        .reset_index()
        .rename(columns={"index": "term_str"})
    )

    VOCAB.index.name = "term_id"

    return TOKENS, VOCAB

In [260]:

TOKENS, VOCAB = clean_text_pickwick(books[title])

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "Pickwick Papers"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])


clean_pick_tokens= TOKENS
clean_pick_vocab = VOCAB

clean_pick_tokens

token_str   term_str
book_title      chap_num para_num sent_num token_num                      
Pickwick Papers 1        0        0        0                The        the
                                           1              first      first
                                           2                ray        ray
                                           3                 of         of
                                           4              light      light
...                                                         ...        ...
                57       32       18       28           nothing    nothing
                                           29               but        but
                                           30             death      death
                                           31              will       will
                                           32         terminate  terminate

[302611 rows x 2 columns]

## Mystery of Edwin Drood

In [261]:
title= "The Mystery of Edwin Drood.txt"

In [262]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+[ivxlcdm]+\.\s*$')


TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "The Mystery of Edwin Drood"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

clean_drood_tokens= TOKENS
clean_drood_vocab = VOCAB



clean_drood_tokens

token_str  \
book_title                 chap_num para_num sent_num token_num             
The Mystery of Edwin Drood 1        0        0        0               THE   
                                                      1              DAWN   
                                    1        0        0                An   
                                                      1           ancient   
                                                      2           English   
...                                                                   ...   
                           23       245      1        1                if   
                                                      2                 I   
                                                      3               was   
                                                      4                to   
                                                      5            deny—’   

                                                                term_str  
book_title                 chap_num para_num sent_num token_num           
The Mystery of Edwin Drood 1        0        0        0              the  
                                                      1             dawn  
                                    1        0        0               an  
                                                      1          ancient  
                                                      2          english  
...                                                                  ...  
                           23       245      1        1               if  
                                                      2                i  
                                                      3              was  
                                                      4               to  
                                                      5             deny  

[96436 rows x 2 columns]

## The Old Curiosity Shop

In [263]:
title= "The Old Curiosity Shop.txt"

In [264]:
TOKENS, VOCAB = clean_text(books[title],chap_pat=r'^\s*chapter\s+\d+\s*$')

tokens_dict[title] = TOKENS
vocab_dict[title] = VOCAB

TOKENS.reset_index(inplace=True)

TOKENS["book_title"]= "The Old Curiosity Shop"

TOKENS = TOKENS.set_index(["book_title", "chap_num", "para_num", "sent_num", "token_num"])


clean_curiosity_tokens= TOKENS
clean_curiosity_vocab = VOCAB

clean_curiosity_tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
                       73       27       0        17                a   
                                                  18             tale   
                                                  19             that   
                                                  20               is   
                                                  21             told   

                                                             term_str  
book_title             chap_num para_num sent_num token_num            
The Old Curiosity Shop 1        0        0        0          although  
                                                  1                 i  
                                                  2                am  
                                                  3                an  
                                                  4               old  
...                                                               ...  
                       73       27       0        17                a  
                                                  18             tale  
                                                  19             that  
                                                  20               is  
                                                  21             told  

[218729 rows x 2 columns]

## Creating TOKEN Table

In [265]:
Corpus_Tokens = pd.concat([clean_curiosity_tokens,
                           clean_drood_tokens,
                           clean_dombey_tokens,
                           clean_nickle_tokens,
                           clean_bleak_tokens,
                           clean_chuzzle_tokens,
                           clean_david_tokens,
                           clean_tale_two_cities_tokens,
                           clean_times_tokens,
                           clean_twist_tokens,
                           clean_pick_tokens,
                           clean_expectations_tokens,
                           clean_dorrit_tokens,
                           clean_mutual_friend_tokens,
                           clean_barnaby_tokens],axis=0)


In [266]:
Corpus_Tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                             term_str  
book_title             chap_num para_num sent_num token_num            
The Old Curiosity Shop 1        0        0        0          although  
                                                  1                 i  
                                                  2                am  
                                                  3                an  
                                                  4               old  
...                                                               ...  
Barnaby Rudge          82       20       1        20          talking  
                                                  21               to  
                                                  22              the  
                                                  23          present  
                                                  24             time  

[3857375 rows x 2 columns]

In [267]:
#nltk.download('averaged_perceptron_tagger')
#nltk.download('averaged_perceptron_tagger_eng')

In [268]:
Corpus_Tokens['pos_tuple'] = nltk.pos_tag(Corpus_Tokens['token_str'].tolist())

In [269]:
Corpus_Tokens['pos'] = Corpus_Tokens.pos_tuple.apply(lambda x: x[1])

In [270]:
Corpus_Tokens['pos_group'] = Corpus_Tokens.pos.str[:2]

In [271]:
Corpus_Tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                             term_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          although   
                                                  1                 i   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                                  pos_tuple  \
book_title             chap_num para_num sent_num token_num                   
The Old Curiosity Shop 1        0        0        0          (Although, IN)   
                                                  1                (I, PRP)   
                                                  2               (am, VBP)   
                                                  3                (an, DT)   
                                                  4               (old, JJ)   
...                                                                     ...   
Barnaby Rudge          82       20       1        20         (talking, VBG)   
                                                  21               (to, TO)   
                                                  22              (the, DT)   
                                                  23          (present, JJ)   
                                                  24             (time, NN)   

                                                             pos pos_group  
book_title             chap_num para_num sent_num token_num                 
The Old Curiosity Shop 1        0        0        0           IN        IN  
                                                  1          PRP        PR  
                                                  2          VBP        VB  
                                                  3           DT        DT  
                                                  4           JJ        JJ  
...                                                          ...       ...  
Barnaby Rudge          82       20       1        20         VBG        VB  
                                                  21          TO        TO  
                                                  22          DT        DT  
                                                  23          JJ        JJ  
                                                  24          NN        NN  

[3857375 rows x 5 columns]

In [272]:
Corpus_Tokens.to_csv("Corpus_Tokens.csv", index=True)

## VOCAB Table

In [273]:
Corpus_Vocab = Corpus_Tokens.term_str.value_counts().to_frame('n').reset_index().rename(columns={'index':'term_str'})
Corpus_Vocab.index.name = 'term_id'
Corpus_Vocab['p'] = Corpus_Vocab.n / Corpus_Vocab.n.sum()

In [274]:
Corpus_Vocab['i'] = -np.log2(Corpus_Vocab.p)

In [275]:
max_pos = (
    Corpus_Tokens[['term_str','pos_group']]
    .value_counts()
    .unstack(fill_value=0)
    .idxmax(axis=1)
)

Corpus_Vocab['max_pos_group'] = Corpus_Vocab['term_str'].map(max_pos)

In [276]:
max_pos = (
    Corpus_Tokens[['term_str', 'pos']]
    .value_counts()
    .unstack(fill_value=0)
    .idxmax(axis=1)
)

Corpus_Vocab['max_pos'] = Corpus_Vocab['term_str'].map(max_pos)

In [277]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/josephhudson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [278]:
sw = pd.DataFrame({'stop': 1}, index=nltk.corpus.stopwords.words('english'))
sw.index.name='term_str'
sw.head()

,stop
term_str,
a,1
about,1
above,1
after,1
again,1


In [279]:
Corpus_Vocab['stop'] = Corpus_Vocab['term_str'].map(sw['stop']).fillna(0).astype(int)

In [280]:
Corpus_Vocab["stop"].value_counts()

stop
0    43406
1      142
Name: count, dtype: int64

In [281]:
p_stemmer = PorterStemmer()

In [282]:
Corpus_Tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                             term_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          although   
                                                  1                 i   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                                  pos_tuple  \
book_title             chap_num para_num sent_num token_num                   
The Old Curiosity Shop 1        0        0        0          (Although, IN)   
                                                  1                (I, PRP)   
                                                  2               (am, VBP)   
                                                  3                (an, DT)   
                                                  4               (old, JJ)   
...                                                                     ...   
Barnaby Rudge          82       20       1        20         (talking, VBG)   
                                                  21               (to, TO)   
                                                  22              (the, DT)   
                                                  23          (present, JJ)   
                                                  24             (time, NN)   

                                                             pos pos_group  
book_title             chap_num para_num sent_num token_num                 
The Old Curiosity Shop 1        0        0        0           IN        IN  
                                                  1          PRP        PR  
                                                  2          VBP        VB  
                                                  3           DT        DT  
                                                  4           JJ        JJ  
...                                                          ...       ...  
Barnaby Rudge          82       20       1        20         VBG        VB  
                                                  21          TO        TO  
                                                  22          DT        DT  
                                                  23          JJ        JJ  
                                                  24          NN        NN  

[3857375 rows x 5 columns]

In [283]:
Corpus_Vocab['p_stem'] = Corpus_Vocab['term_str'].apply(p_stemmer.stem)

In [284]:
OHCO = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [285]:
bag = 'BOOKS'

In [286]:
BOW = Corpus_Tokens.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 

In [287]:
DTCM = BOW.n.unstack(fill_value=0)

In [288]:
DF = DTCM.astype('bool').sum() 

In [289]:
N = DTCM.shape[0]

In [290]:
IDF = np.log2(N / DF)

In [291]:
dfidf=DF*IDF

In [292]:
dfidf

term_str
0          4.523562
000        4.523562
1         11.008169
12         4.523562
12d        4.523562
            ...    
zone       7.047124
zoo        4.523562
zoodle     4.523562
zooks      4.523562
zounds     4.523562
Length: 43548, dtype: float64

In [293]:
Corpus_Vocab['dfidf'] = Corpus_Vocab['term_str'].map(dfidf)

In [294]:
Corpus_Vocab

,term_str,n,p,i,max_pos_group,max_pos,stop,p_stem,dfidf
term_id,,,,,,,,,
0,the,182824,4.739596e-02,4.399092,DT,DT,1,the,0.000000
1,and,136204,3.531002e-02,4.823778,CC,CC,1,and,0.000000
2,to,103287,2.677650e-02,5.222889,TO,TO,1,to,0.000000
3,of,98236,2.546706e-02,5.295224,IN,IN,1,of,0.000000
4,a,85297,2.211271e-02,5.498981,DT,DT,1,a,0.000000
...,...,...,...,...,...,...,...,...,...
43543,charons,1,2.592437e-07,21.879188,NN,NNP,0,charon,4.523562
43544,nowhaving,1,2.592437e-07,21.879188,VB,VBG,0,nowhav,4.523562
43545,erections,1,2.592437e-07,21.879188,NN,NNS,0,erect,4.523562


In [295]:
Corpus_Vocab.reset_index().set_index('term_str')['dfidf'].nlargest(20)

term_str
kit           12.188496
bucket        12.188496
charley       12.188496
sawyer        12.188496
edward        12.188496
toms          12.188496
jingle        12.188496
monks         12.188496
david         12.188496
chancery      12.188496
colonel       12.188496
sloppy        12.188496
pony          12.188496
mills         12.188496
emma          12.188496
deary         12.188496
chancellor    12.188496
daniel        12.188496
repeats       12.188496
citizen       12.188496
Name: dfidf, dtype: float64

In [296]:
Corpus_Vocab.to_csv("Corpus_Vocab.csv", index=True)

## LIB Table

In [297]:
folder_path = '/Users/josephhudson/Desktop/DS 5001 Project/Dickens Corpus/'
book_data = []

for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)
        
        with open(file_path, 'r', encoding='utf-8') as file:     
            book_data.append({
                'File Name': filename,
                'Title': filename.removesuffix('.txt'),
                'Path': file_path
            })



Corpus_Lib = pd.DataFrame(book_data)

In [298]:
Corpus_Tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                             term_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          although   
                                                  1                 i   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                                  pos_tuple  \
book_title             chap_num para_num sent_num token_num                   
The Old Curiosity Shop 1        0        0        0          (Although, IN)   
                                                  1                (I, PRP)   
                                                  2               (am, VBP)   
                                                  3                (an, DT)   
                                                  4               (old, JJ)   
...                                                                     ...   
Barnaby Rudge          82       20       1        20         (talking, VBG)   
                                                  21               (to, TO)   
                                                  22              (the, DT)   
                                                  23          (present, JJ)   
                                                  24             (time, NN)   

                                                             pos pos_group  
book_title             chap_num para_num sent_num token_num                 
The Old Curiosity Shop 1        0        0        0           IN        IN  
                                                  1          PRP        PR  
                                                  2          VBP        VB  
                                                  3           DT        DT  
                                                  4           JJ        JJ  
...                                                          ...       ...  
Barnaby Rudge          82       20       1        20         VBG        VB  
                                                  21          TO        TO  
                                                  22          DT        DT  
                                                  23          JJ        JJ  
                                                  24          NN        NN  

[3857375 rows x 5 columns]

In [299]:
Corpus_Lib=pd.DataFrame(Corpus_Tokens.reset_index().book_title.unique())

In [300]:
Corpus_Lib.columns=['book_title']

In [301]:
Corpus_Lib['author'] = 'Charles Dickens'

In [302]:
book_lengths = Corpus_Tokens.groupby('book_title').term_str.count()

Corpus_Lib['book_len'] = Corpus_Lib['book_title'].map(book_lengths)

In [303]:
chap_counts = (
    Corpus_Tokens.index.to_frame(index=False)
    .groupby('book_title')['chap_num']
    .nunique()
)

Corpus_Lib['n_chaps'] = Corpus_Lib['book_title'].map(chap_counts)


In [304]:
token_counts = Corpus_Tokens.groupby('book_title').size()

sent_counts = (
    Corpus_Tokens.reset_index()
    [['book_title','chap_num','para_num','sent_num']]
    .drop_duplicates()
    .groupby('book_title')
    .size()
)

avg_sent_len = token_counts / sent_counts

In [305]:
Corpus_Lib['avg_sent_len'] = Corpus_Lib['book_title'].map(avg_sent_len)

In [306]:
Corpus_Lib

,book_title,author,book_len,n_chaps,avg_sent_len
0,The Old Curiosity Shop,Charles Dickens,218729,73,17.999424
1,The Mystery of Edwin Drood,Charles Dickens,96436,23,12.012456
2,Dombey and Son,Charles Dickens,356587,62,15.899189
3,Nicholas Nickleby,Charles Dickens,324259,65,14.234997
4,Bleak House,Charles Dickens,354643,67,13.623872
5,Martin Chuzzlewit,Charles Dickens,338838,55,13.576872
6,David Copperfield,Charles Dickens,357805,64,13.334513
7,A Tale of Two Cities 1,Charles Dickens,17520,6,15.077453
8,A Tale of Two Cities 2,Charles Dickens,70973,24,13.998619
9,A Tale of Two Cities 3,Charles Dickens,48583,15,13.394817


In [307]:
Corpus_Lib.to_csv('Corpus_Lib.csv', index=False)